In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

RANDOM_STATE = 42

In [4]:
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "raw"

X = pd.read_csv(
    DATA_DIR / "secom.data",
    sep=r"\s+",
    header=None
)

labels = pd.read_csv(
    DATA_DIR / "secom_labels.data",
    sep=r"\s+",
    header=None
)

# 기존 라벨 유지: 양품 -1, 불량 1
y = labels.iloc[:, 0].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nLabel distribution")
print(y.value_counts())

X shape: (1567, 590)
y shape: (1567,)

Label distribution
0
-1    1463
 1     104
Name: count, dtype: int64


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

Train shape: (1253, 590)
Test shape : (314, 590)


In [6]:
train_missing_ratio = X_train.isna().mean()

high_missing_cols = train_missing_ratio[
    train_missing_ratio >= 0.5
].index.tolist()

X_train_hm = X_train.drop(columns=high_missing_cols)
X_test_hm = X_test.drop(columns=high_missing_cols)

print("제거한 고결측 변수:", len(high_missing_cols))
print("남은 변수:", X_train_hm.shape[1])

제거한 고결측 변수: 24
남은 변수: 566


In [7]:
train_unique_counts = X_train_hm.nunique(dropna=True)

constant_cols = train_unique_counts[
    train_unique_counts <= 1
].index.tolist()

X_train_clean = X_train_hm.drop(columns=constant_cols)
X_test_clean = X_test_hm.drop(columns=constant_cols)

print("제거한 상수 변수:", len(constant_cols))
print("남은 변수:", X_train_clean.shape[1])

제거한 상수 변수: 116
남은 변수: 450


In [8]:
imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train_clean)
X_test_imputed = imputer.transform(X_test_clean)

print("Train 결측값:", np.isnan(X_train_imputed).sum())
print("Test 결측값 :", np.isnan(X_test_imputed).sum())

Train 결측값: 0
Test 결측값 : 0


In [9]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Train shape:", X_train_scaled.shape)
print("Test shape :", X_test_scaled.shape)

Train shape: (1253, 450)
Test shape : (314, 450)


In [ ]:
bayes_model = GaussianNB()

bayes_model.fit(
    X_train_scaled,
    y_train
)

print("베이지안 분류기 학습 완료")
print("클래스 순서:", bayes_model.classes_)
print("학습한 사전확률:", bayes_model.class_prior_)

베이지안 분류기 학습 완료
클래스 순서: [-1  1]
학습한 사전확률: [0.93375898 0.06624102]


In [11]:
# 양품·불량 예측
bayes_pred = bayes_model.predict(X_test_scaled)

# 불량(1)에 해당하는 예측 확률
fail_idx = list(bayes_model.classes_).index(1)
bayes_prob = bayes_model.predict_proba(X_test_scaled)[:, fail_idx]

# 평가
bayes_bal_acc = balanced_accuracy_score(y_test, bayes_pred)
bayes_auc = roc_auc_score(
    (y_test == 1).astype(int),
    bayes_prob
)

print("=== Gaussian Naive Bayes ===")
print(f"Balanced Accuracy: {bayes_bal_acc:.4f}")
print(f"ROC-AUC          : {bayes_auc:.4f}")

print("\nClassification Report")
print(classification_report(
    y_test,
    bayes_pred,
    labels=[-1, 1],
    target_names=["Pass (-1)", "Fail (+1)"],
    digits=4,
    zero_division=0
))

print("\nConfusion Matrix")
print("행: 실제 정답 / 열: 예측 결과")
print("순서: 양품(-1), 불량(+1)")
print(confusion_matrix(
    y_test,
    bayes_pred,
    labels=[-1, 1]
))

=== Gaussian Naive Bayes ===
Balanced Accuracy: 0.5734
ROC-AUC          : 0.5870

Classification Report
              precision    recall  f1-score   support

   Pass (-1)     1.0000    0.1468    0.2560       293
   Fail (+1)     0.0775    1.0000    0.1438        21

    accuracy                         0.2038       314
   macro avg     0.5387    0.5734    0.1999       314
weighted avg     0.9383    0.2038    0.2485       314


Confusion Matrix
행: 실제 정답 / 열: 예측 결과
순서: 양품(-1), 불량(+1)
[[ 43 250]
 [  0  21]]


In [15]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.metrics import precision_score, recall_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

# 기본값 1e-9부터 1까지, 10개 후보
smoothing_values = [0.1, 0.3, 0.5, 1.0, 2.0, 3.0, 5.0, 10.0]
cv_rows = []

for train_pos, valid_pos in cv.split(X_train, y_train):
    X_fit = X_train.iloc[train_pos]
    X_valid = X_train.iloc[valid_pos]
    y_fit = y_train.iloc[train_pos]
    y_valid = y_train.iloc[valid_pos]

    # 이번 학습 분할에서 사용할 변수 선택
    keep_cols = X_fit.columns[
        (X_fit.isna().mean() < 0.5)
        & (X_fit.nunique(dropna=True) > 1)
    ]

    prep = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler()
    )

    X_fit_ready = prep.fit_transform(X_fit[keep_cols])
    X_valid_ready = prep.transform(X_valid[keep_cols])

    for smoothing in smoothing_values:
        model = GaussianNB(var_smoothing=smoothing)
        model.fit(X_fit_ready, y_fit)
        pred = model.predict(X_valid_ready)

        cv_rows.append({
            "var_smoothing": smoothing,
            "bal_acc": balanced_accuracy_score(y_valid, pred),
            "recall": recall_score(y_valid, pred, pos_label=1),
            "precision": precision_score(
                y_valid, pred, pos_label=1, zero_division=0
            )
        })

# 후보별 5개 검증 결과 요약
cv_table = (
    pd.DataFrame(cv_rows)
    .groupby("var_smoothing", as_index=False)
    .agg(
        CV_Balanced_Accuracy=("bal_acc", "mean"),
        CV_Std=("bal_acc", "std"),
        CV_Fail_Recall=("recall", "mean"),
        CV_Fail_Precision=("precision", "mean")
    )
    .sort_values("CV_Balanced_Accuracy", ascending=False)
    .reset_index(drop=True)
)

best_smoothing = float(cv_table.loc[0, "var_smoothing"])

print(cv_table.to_string(
    index=False,
    float_format=lambda value: f"{value:.4g}"
))
print(f"\n선택된 var_smoothing: {best_smoothing:.0e}")

 var_smoothing  CV_Balanced_Accuracy  CV_Std  CV_Fail_Recall  CV_Fail_Precision
           0.5                0.5674 0.07064          0.4809             0.1011
             1                0.5666  0.0439          0.3478             0.1236
             2                 0.556 0.03592          0.2051              0.171
           0.3                0.5554 0.06296          0.5544            0.08846
           0.1                0.5368 0.05804          0.6515             0.0749
             3                0.5267 0.01603          0.1081             0.1399
             5                0.5131 0.02248         0.06029             0.1102
            10                0.5128 0.01495         0.04779             0.1456

선택된 var_smoothing: 5e-01


In [17]:
tuned_bayes = GaussianNB(var_smoothing=best_smoothing)
tuned_bayes.fit(X_train_scaled, y_train)

tuned_pred = tuned_bayes.predict(X_test_scaled)

fail_idx = list(tuned_bayes.classes_).index(1)
tuned_prob = tuned_bayes.predict_proba(X_test_scaled)[:, fail_idx]

print("=== Tuned Gaussian Naive Bayes ===")
print(f"var_smoothing: {best_smoothing:.0e}")
print(
    "Balanced Accuracy:",
    round(balanced_accuracy_score(y_test, tuned_pred), 4)
)
print(
    "ROC-AUC:",
    round(roc_auc_score((y_test == 1).astype(int), tuned_prob), 4)
)

print("\nClassification Report")
print(classification_report(
    y_test,
    tuned_pred,
    labels=[-1, 1],
    target_names=["Pass (-1)", "Fail (+1)"],
    digits=4,
    zero_division=0
))

print("\nConfusion Matrix")
print(confusion_matrix(
    y_test,
    tuned_pred,
    labels=[-1, 1]
))

=== Tuned Gaussian Naive Bayes ===
var_smoothing: 5e-01
Balanced Accuracy: 0.6797
ROC-AUC: 0.729

Classification Report
              precision    recall  f1-score   support

   Pass (-1)     0.9667    0.6928    0.8072       293
   Fail (+1)     0.1346    0.6667    0.2240        21

    accuracy                         0.6911       314
   macro avg     0.5506    0.6797    0.5156       314
weighted avg     0.9110    0.6911    0.7682       314


Confusion Matrix
[[203  90]
 [  7  14]]
